In [ ]:
import numpy as np
import seaborn as sns
from ising.model import FitMethod

from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from ising import Ising

RANDOM_SEED = 202605211526

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, name="reduced_no_imputation", with_imputation=False)
_, Y0, _ = dataset.indices_to_numpy(
    kind="time-series", binarise=False, seed=RANDOM_SEED
)
_, Y, X = dataset.indices_to_numpy(
    kind="time-series", binarise=True, scale=0.25, seed=RANDOM_SEED
)
node_labels = np.asarray(dataset.schema.get_short_names(kind="measurement"))

In [ ]:
model = Ising.fit(
    Y,
    X=X,
    method=FitMethod.TIME_SERIES,
    node_labels=node_labels,
    rng=RANDOM_SEED,
    self_loops=True,
)

In [ ]:
init_energy = np.empty(Y.shape[0], dtype=np.float64)
final_energy = np.empty(Y.shape[0], dtype=np.float64)

spins_changed = np.empty(Y.shape[0], dtype=np.int64)
for i in range(Y.shape[0]):
    init_energy[i] = -(
        model.parallel_glauber_theta(
            Y[i, 0], np.concat(([1], X[i, 0])), model.h, model.j, model.adj
        )
        @ Y[i, 0]
    )
    final_energy[i] = -(
        model.parallel_glauber_theta(
            Y[i, 1], np.concat(([1], X[i, 1])), model.h, model.j, model.adj
        )
        @ Y[i, 1]
    )
    spins_changed[i] = (np.abs(Y[i, 1] - Y[i, 0]) // 2).sum()

energy_change = final_energy - init_energy

In [ ]:
model.node_labels

In [ ]:
Y[1525, 0]

In [ ]:
Y[1525, 1]

In [ ]:
Y0[1525, 0]

In [ ]:
Y0[1525, 1]

In [ ]:
Y[-2, 0]

In [ ]:
init_energy.argmax()

In [ ]:
sns.relplot(x=init_energy, y=spins_changed)

In [ ]:
sns.relplot(x=init_energy, y=final_energy)

In [ ]:
np.concat(([1], X[0, 0]))